# Chapter 2 — The Correlation: Who runs the best facilities?

**Narrative:** After seeing the map, the viewer asks *why* quality differs across regions. The answer is ownership — government and NFP facilities consistently outperform private ones. The geography story is really an ownership story.

**Visuals:**
1. Scatter — access_rate vs avg_quality (MMM colour)
2. Scatter with year slider (2023–2024)
3. Bar — org type breakdown (count + funding)
4. Bar — quality by ownership type over time
5. Bar — quality by MMM remoteness
6. Bar — sub-rating breakdown by ownership type
7. Line — quality trend over time by ownership (quarterly)
8. Box — quality distribution by ownership type
9. Bar — funding per facility by org type

**Source files:** `star_ratings_by_facility.csv`, `service_users_by_sa3.csv`, `abs_population_by_sa3.csv`, `service_funding_by_facility.csv`

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

DATA = '../../data/clean/'

## 1. Load & clean

In [2]:
ratings_raw = pd.read_csv(DATA + 'star_ratings_by_facility.csv')
users       = pd.read_csv(DATA + 'service_users_by_sa3.csv')
pop         = pd.read_csv(DATA + 'abs_population_by_sa3.csv')
funding     = pd.read_csv(DATA + 'service_funding_by_facility.csv')

# Extract year from snapshot_date and normalise Purpose casing
ratings_raw['snapshot_date'] = pd.to_datetime(ratings_raw['snapshot_date'])
ratings_raw['year'] = ratings_raw['snapshot_date'].dt.year
ratings_raw['purpose_clean'] = ratings_raw['Purpose'].str.strip().str.lower().map({
    'for profit':     'For-Profit',
    'not for profit': 'Not-for-Profit',
    'government':     'Government',
})

# Coerce sa3_code to nullable int for consistent joins
for df in [ratings_raw, users, pop, funding]:
    if 'sa3_code' in df.columns:
        df['sa3_code'] = pd.to_numeric(df['sa3_code'], errors='coerce').astype('Int64')

print('Ratings years:', sorted(ratings_raw['year'].unique()))
print('Users years:  ', sorted(users['year'].unique()))
print('Pop years:    ', sorted(pop['year'].unique()))
print('Funding years:', sorted(funding['year'].unique()))

Ratings years: [np.int32(2023), np.int32(2024), np.int32(2025), np.int32(2026)]
Users years:   [np.int64(2023), np.int64(2024), np.int64(2025)]
Pop years:     [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Funding years: [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


## 2. Build SA3 × year base dataframe

Overlap years across all three key files: **2023–2024**.

In [3]:
quality_sa3 = (
    ratings_raw
    .groupby(['sa3_code', 'sa3_name', 'year'])
    .agg(
        avg_quality  = ('quality_score', 'mean'),
        mmm_code     = ('mmm_code', lambda x: x.mode().iloc[0]),
        state        = ('state', lambda x: x.mode().iloc[0]),
        n_facilities = ('Service Name', 'nunique'),
    )
    .reset_index()
)

df = (
    users[['sa3_code', 'year', 'total_residential', 'total_homecare']]
    .merge(pop[['sa3_code', 'year', 'pop_65_plus']], on=['sa3_code', 'year'])
)
df['access_rate']          = df['total_residential'] / df['pop_65_plus'] * 100
df['total_users']          = df['total_residential'] + df['total_homecare']
df['combined_access_rate'] = df['total_users'] / df['pop_65_plus'] * 100
df = df.merge(quality_sa3, on=['sa3_code', 'year'])
df['care_gap_index'] = df['access_rate'] / df['avg_quality']
df = df[df['year'].isin([2023, 2024])].copy()

print(f'{len(df)} rows | {df["sa3_code"].nunique()} unique SA3s | years: {sorted(df["year"].unique())}')
df.head(3)

646 rows | 323 unique SA3s | years: [np.int64(2023), np.int64(2024)]


,sa3_code,year,total_residential,total_homecare,pop_65_plus,access_rate,total_users,combined_access_rate,sa3_name,avg_quality,mmm_code,state,n_facilities,care_gap_index
0,10102,2024,262.0,497,9399.0,2.787531,759.0,8.075327,Queanbeyan,3.562500,MM1,NSW,3,0.782465
1,10103,2024,77.0,156,4432.0,1.737365,233.0,5.257220,Snowy Mountains,3.708333,MM4,NSW,2,0.468503
2,10104,2024,909.0,1826,25219.0,3.604425,2735.0,10.844998,South Coast,3.673077,MM4,NSW,13,0.981309


## 3. Shared constants

In [4]:
mmm_order = ['MM1', 'MM2', 'MM3', 'MM4', 'MM5', 'MM6', 'MM7']
mmm_labels = {
    'MM1': 'MM1 Major city',
    'MM2': 'MM2 Inner regional',
    'MM3': 'MM3 Outer regional',
    'MM4': 'MM4 Remote',
    'MM5': 'MM5 Small rural town',
    'MM6': 'MM6 Remote community',
    'MM7': 'MM7 Very remote',
}
purpose_order  = ['Government', 'Not-for-Profit', 'For-Profit']
colour_purpose = {'Government': '#2166ac', 'Not-for-Profit': '#4dac26', 'For-Profit': '#d01c8b'}
colour_org     = {'government': '#2166ac', 'not_for_profit': '#4dac26', 'profit': '#d01c8b'}
label_org      = {'government': 'Government', 'not_for_profit': 'Not-for-Profit', 'profit': 'For-Profit'}

# Headline numbers (used across multiple cells)
headline = (
    ratings_raw
    .dropna(subset=['purpose_clean', 'quality_score'])
    .groupby('purpose_clean')['quality_score']
    .mean()
    .sort_values(ascending=False)
)
mmm_quality = (
    ratings_raw
    .dropna(subset=['mmm_code', 'quality_score'])
    .groupby('mmm_code')['quality_score']
    .mean()
    .sort_index()
)
print('Quality by ownership:', headline.round(3).to_dict())
print('Quality by MMM:      ', mmm_quality.round(3).to_dict())

Quality by ownership: {'Government': 4.071, 'Not-for-Profit': 3.609, 'For-Profit': 3.498}
Quality by MMM:       {'MM1': 3.571, 'MM2': 3.571, 'MM3': 3.524, 'MM4': 3.628, 'MM5': 3.875, 'MM6': 3.77, 'MM7': 3.949}


## Visual 1 — Scatter: Access Rate vs Quality (2024, MMM colour)

In [5]:
df_2024 = df[df['year'] == 2024].copy()
df_2024['mmm_label'] = df_2024['mmm_code'].map(mmm_labels)

fig1 = px.scatter(
    df_2024,
    x='access_rate',
    y='avg_quality',
    size='pop_65_plus',
    color='mmm_code',
    category_orders={'mmm_code': mmm_order},
    color_discrete_sequence=px.colors.sequential.Plasma_r[:7],
    hover_name='sa3_name',
    hover_data={
        'state': True, 'mmm_code': False, 'mmm_label': True,
        'access_rate': ':.1f', 'avg_quality': ':.2f',
        'pop_65_plus': ':,', 'n_facilities': True,
    },
    trendline='ols',
    labels={
        'access_rate': 'Access Rate (% of 65+ in residential care)',
        'avg_quality': 'Quality Score (avg Star Rating)',
        'mmm_code': 'Remoteness', 'pop_65_plus': 'Pop 65+',
        'mmm_label': 'Remoteness', 'n_facilities': 'Facilities',
    },
    title='Access Rate vs Quality Score by SA3 (2024)',
    size_max=40,
)
fig1.update_layout(height=600)
fig1.show()

## Visual 2 — Scatter with Year Slider (2023–2024)

In [6]:
df_slider = df.copy()
df_slider['mmm_label'] = df_slider['mmm_code'].map(mmm_labels)

fig2 = px.scatter(
    df_slider,
    x='access_rate',
    y='avg_quality',
    size='pop_65_plus',
    color='mmm_code',
    animation_frame='year',
    animation_group='sa3_code',
    category_orders={'mmm_code': mmm_order, 'year': [2023, 2024]},
    color_discrete_sequence=px.colors.sequential.Plasma_r[:7],
    hover_name='sa3_name',
    hover_data={
        'state': True, 'mmm_label': True, 'mmm_code': False, 'year': False,
        'access_rate': ':.1f', 'avg_quality': ':.2f', 'pop_65_plus': ':,',
    },
    labels={
        'access_rate': 'Access Rate (% of 65+ in residential care)',
        'avg_quality': 'Quality Score (avg Star Rating)',
        'mmm_code': 'Remoteness', 'mmm_label': 'Remoteness', 'pop_65_plus': 'Pop 65+',
    },
    range_x=[0, df_slider['access_rate'].quantile(0.99) * 1.1],
    range_y=[2.5, 5.0],
    title='Access Rate vs Quality Score by SA3 — 2023 to 2024',
    size_max=40,
)
fig2.update_layout(height=600)
fig2.show()

## Visual 3 — Org Type Breakdown: Facility Count and Funding (2024)

In [7]:
funding_2024 = funding[funding['year'] == 2024].copy()
org_summary = (
    funding_2024
    .groupby('org_type')
    .agg(n_facilities=('service_name', 'nunique'), total_funding=('funding', 'sum'))
    .reset_index()
    .sort_values('n_facilities', ascending=True)
)
org_summary['funding_bn'] = org_summary['total_funding'] / 1e9

fig3 = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Number of Facilities', 'Total Government Funding ($B)'],
    shared_yaxes=True,
)
for _, row in org_summary.iterrows():
    label  = label_org.get(row['org_type'], row['org_type'])
    colour = colour_org.get(row['org_type'], '#888')
    fig3.add_trace(
        go.Bar(x=[row['n_facilities']], y=[label], orientation='h',
               marker_color=colour, name=label, showlegend=False,
               text=[f"{row['n_facilities']:,}"], textposition='auto'),
        row=1, col=1
    )
    fig3.add_trace(
        go.Bar(x=[row['funding_bn']], y=[label], orientation='h',
               marker_color=colour, name=label, showlegend=False,
               text=[f"${row['funding_bn']:.1f}B"], textposition='auto'),
        row=1, col=2
    )
fig3.update_layout(title='Aged Care Providers by Org Type (2024)', height=300,
                   xaxis_title='Facilities', xaxis2_title='Funding ($B)')
fig3.show()

## Visual 4 — Quality by Ownership Type Over Time

In [8]:
ownership_yr = (
    ratings_raw
    .dropna(subset=['purpose_clean', 'quality_score'])
    .groupby(['purpose_clean', 'year'])
    .agg(avg_quality=('quality_score', 'mean'), n_facilities=('Service Name', 'nunique'))
    .reset_index()
)

fig4 = px.bar(
    ownership_yr,
    x='year', y='avg_quality', color='purpose_clean', barmode='group',
    category_orders={'purpose_clean': purpose_order},
    color_discrete_map=colour_purpose,
    text_auto='.2f',
    hover_data={'n_facilities': True},
    labels={'avg_quality': 'Avg Quality Score', 'purpose_clean': 'Ownership',
            'year': 'Year', 'n_facilities': 'Facilities'},
    title='Quality Score by Ownership Type Over Time',
)
fig4.update_layout(height=450, yaxis_range=[2.5, 5.0])
fig4.show()

## Visual 5 — Ownership Quality Within Each MMM Band

Controls for geography: even within the same remoteness band, government outperforms for-profit. Kills the "rural facilities just happen to be government-run" counter-argument.

In [9]:
ownership_mmm = (
    ratings_raw
    .dropna(subset=['purpose_clean', 'mmm_code', 'quality_score'])
    .groupby(['mmm_code', 'purpose_clean'])
    .agg(avg_quality=('quality_score', 'mean'), n=('Service Name', 'nunique'))
    .reset_index()
)
ownership_mmm['mmm_label'] = ownership_mmm['mmm_code'].map(mmm_labels)

fig5 = px.bar(
    ownership_mmm,
    x='mmm_code', y='avg_quality', color='purpose_clean',
    barmode='group',
    category_orders={
        'mmm_code':      mmm_order,
        'purpose_clean': purpose_order,
    },
    color_discrete_map=colour_purpose,
    text_auto='.2f',
    hover_data={'n': True, 'mmm_label': True, 'mmm_code': False},
    labels={
        'mmm_code':      'Remoteness (MMM)',
        'avg_quality':   'Avg Quality Score',
        'purpose_clean': 'Ownership',
        'n':             'Facilities',
        'mmm_label':     'Remoteness',
    },
    title='Ownership Gap Persists Across All Remoteness Bands — it\'s not just geography',
)
fig5.update_layout(height=480, yaxis_range=[2.0, 5.0])
fig5.show()

## Visual 6 — Sub-rating Breakdown by Ownership Type

Shows which of the 4 dimensions drives the ownership quality gap.

In [10]:
subratings = ['residents_exp', 'staffing', 'compliance', 'quality_measures']
sublabels  = {
    'residents_exp':    'Residents\' Experience',
    'staffing':         'Staffing',
    'compliance':       'Compliance',
    'quality_measures': 'Quality Measures',
}

subrating_data = (
    ratings_raw
    .dropna(subset=['purpose_clean'] + subratings)
    .groupby('purpose_clean')[subratings]
    .mean()
    .reset_index()
    .melt(id_vars='purpose_clean', var_name='subrating', value_name='avg_score')
)
subrating_data['subrating_label'] = subrating_data['subrating'].map(sublabels)

fig6 = px.bar(
    subrating_data,
    x='subrating_label', y='avg_score', color='purpose_clean', barmode='group',
    category_orders={
        'purpose_clean': purpose_order,
        'subrating_label': list(sublabels.values()),
    },
    color_discrete_map=colour_purpose,
    text_auto='.2f',
    labels={
        'subrating_label': 'Star Rating Dimension',
        'avg_score':       'Avg Score',
        'purpose_clean':   'Ownership',
    },
    title='Sub-rating Breakdown by Ownership Type — where does the gap come from?',
)
fig6.update_layout(height=480, yaxis_range=[2.0, 5.0])
fig6.show()

## Visual 7 — Quality Trend Over Time by Ownership (Quarterly)

Full quarterly series (May 2023 → Feb 2026). Shows whether the ownership gap is narrowing or holding post-mandate.

In [11]:
trend_data = (
    ratings_raw
    .dropna(subset=['purpose_clean', 'quality_score'])
    .groupby(['purpose_clean', 'snapshot_date'])
    .agg(avg_quality=('quality_score', 'mean'), n_facilities=('Service Name', 'nunique'))
    .reset_index()
    .sort_values('snapshot_date')
)

fig7 = px.line(
    trend_data,
    x='snapshot_date', y='avg_quality', color='purpose_clean',
    category_orders={'purpose_clean': purpose_order},
    color_discrete_map=colour_purpose,
    markers=True,
    hover_data={'n_facilities': True},
    labels={
        'snapshot_date': 'Snapshot',
        'avg_quality':   'Avg Quality Score',
        'purpose_clean': 'Ownership',
        'n_facilities':  'Facilities',
    },
    title='Quality Score Trend by Ownership (May 2023 – Feb 2026)',
)
# add_vline doesn't handle string dates on datetime axes in all Plotly versions
fig7.add_shape(
    type='line',
    x0='2023-10-01', x1='2023-10-01',
    y0=0, y1=1, yref='paper',
    line=dict(dash='dash', color='grey', width=1.5),
)
fig7.add_annotation(
    x='2023-10-01', y=1, yref='paper',
    text='Mandate (Oct 2023)',
    showarrow=False, xanchor='left', yanchor='bottom',
    font=dict(color='grey', size=11),
)
fig7.update_layout(height=480, yaxis_range=[2.0, 5.0])
fig7.show()

## Visual 8 — Quality Distribution by Ownership Type (Box Plot)

Averages hide spread. This shows whether for-profit facilities are more variable — some good, many bad — vs government being consistently high.

In [12]:
box_data = ratings_raw.dropna(subset=['purpose_clean', 'quality_score']).copy()

fig8 = px.box(
    box_data,
    x='purpose_clean', y='quality_score', color='purpose_clean',
    category_orders={'purpose_clean': purpose_order},
    color_discrete_map=colour_purpose,
    points='outliers',
    labels={
        'purpose_clean':  'Ownership',
        'quality_score':  'Quality Score (facility × snapshot)',
    },
    title='Quality Score Distribution by Ownership — spread matters as much as averages',
)
fig8.update_layout(height=480, showlegend=False, yaxis_range=[0.5, 5.5])
fig8.show()

## Visual 9 — Funding per Facility by Org Type (2024)

For-profit receives the most funding per facility yet scores lowest on quality — the efficiency paradox.

In [13]:
funding_per_facility = org_summary.copy()
funding_per_facility['funding_per_facility_m'] = (
    funding_per_facility['total_funding'] / funding_per_facility['n_facilities'] / 1e6
)
funding_per_facility['label'] = funding_per_facility['org_type'].map(label_org)
funding_per_facility['colour'] = funding_per_facility['org_type'].map(colour_org)
funding_per_facility = funding_per_facility.sort_values('funding_per_facility_m', ascending=True)

# Add quality score for annotation
quality_lookup = {
    'Government':     headline.get('Government', None),
    'Not-for-Profit': headline.get('Not-for-Profit', None),
    'For-Profit':     headline.get('For-Profit', None),
}
funding_per_facility['avg_quality'] = funding_per_facility['label'].map(quality_lookup)

fig9 = go.Figure()
for _, row in funding_per_facility.iterrows():
    fig9.add_trace(go.Bar(
        x=[row['funding_per_facility_m']],
        y=[row['label']],
        orientation='h',
        marker_color=row['colour'],
        name=row['label'],
        showlegend=False,
        text=[f"${row['funding_per_facility_m']:.2f}M | quality {row['avg_quality']:.2f}"],
        textposition='auto',
        hovertemplate=(
            f"<b>{row['label']}</b><br>"
            f"Funding per facility: ${row['funding_per_facility_m']:.2f}M<br>"
            f"Avg quality score: {row['avg_quality']:.2f}<br>"
            f"Facilities: {row['n_facilities']:,}<extra></extra>"
        ),
    ))

fig9.update_layout(
    title='Funding per Facility by Org Type (2024) — more money, less quality?',
    xaxis_title='Avg Government Funding per Facility ($M)',
    height=300,
)
fig9.show()

## Insight Summary

In [14]:
gap   = headline['Government'] - headline['For-Profit']
mm1_q = mmm_quality.get('MM1', float('nan'))
mm7_q = mmm_quality.get('MM7', float('nan'))

print('=== CHAPTER 2 KEY NUMBERS ===')
print(f"Government avg quality:     {headline['Government']:.2f}")
print(f"Not-for-Profit avg quality: {headline['Not-for-Profit']:.2f}")
print(f"For-Profit avg quality:     {headline['For-Profit']:.2f}")
print(f"Ownership gap (Gov - FP):   {gap:.2f} pts")
print()
print(f"MM1 (Major city) avg quality:  {mm1_q:.2f}")
print(f"MM7 (Very remote) avg quality: {mm7_q:.2f}")
print()

# For-profit share by MMM
fp_by_mmm = (
    ratings_raw
    .dropna(subset=['purpose_clean', 'mmm_code'])
    .groupby(['mmm_code', 'purpose_clean'])['Service Name'].nunique()
    .unstack(fill_value=0)
)
fp_by_mmm['total'] = fp_by_mmm.sum(axis=1)
fp_by_mmm['pct_for_profit'] = fp_by_mmm.get('For-Profit', 0) / fp_by_mmm['total'] * 100
print('For-profit share by MMM:')
print(fp_by_mmm[['For-Profit', 'total', 'pct_for_profit']].round(1))
print()

# Funding per facility
print('Funding per facility (2024):')
for _, row in funding_per_facility.iterrows():
    print(f"  {row['label']:15s} ${row['funding_per_facility_m']:.2f}M  "
          f"| quality {row['avg_quality']:.2f}")

=== CHAPTER 2 KEY NUMBERS ===
Government avg quality:     4.07
Not-for-Profit avg quality: 3.61
For-Profit avg quality:     3.50
Ownership gap (Gov - FP):   0.57 pts

MM1 (Major city) avg quality:  3.57
MM7 (Very remote) avg quality: 3.95

For-profit share by MMM:
purpose_clean  For-Profit  total  pct_for_profit
mmm_code                                        
MM1                   865   1985            43.6
MM2                    69    240            28.7
MM3                    56    258            21.7
MM4                    40    222            18.0
MM5                    32    359             8.9
MM6                     0     30             0.0
MM7                     0     12             0.0

Funding per facility (2024):
  Government      $2.85M  | quality 4.07
  Not-for-Profit  $5.54M  | quality 3.61
  For-Profit      $7.46M  | quality 3.50


---

## Narrative Findings

### The headline
Who runs the facility matters more than where it is. Government-run facilities average **4.07 stars** vs for-profit at **3.50** — a **0.57-point gap** that holds across every remoteness band in Australia. The geography story is really an ownership story.

---

### Finding 1 — The ownership gap is real, and it's not a geography artefact
A common assumption: government facilities score higher because they're concentrated in rural areas where competition is low and the resident mix is different. The data rejects this.

Across every MMM band — from MM1 major cities to MM5 small rural towns — government outperforms for-profit:
- **MM1 (major city):** Government 4.03 vs For-Profit 3.50 — a 0.53-point gap in the most competitive, most privatised market in the country
- **MM5 (small rural):** Government 4.19 vs For-Profit 3.63 — the gap *widens* in rural areas, not narrows
- **MM6 & MM7 (remote/very remote):** No for-profit facilities operate here at all

The ownership gap is not explained by location. It exists within each location.

---

### Finding 2 — Staffing is where the gap lives
Breaking the composite quality score into its 4 sub-dimensions reveals where the gap comes from:

| Dimension | Government | For-Profit | Gap |
|-----------|-----------|------------|-----|
| Staffing | 4.39 | 2.61 | **−1.78 pts** |
| Residents' Experience | 3.67 | 3.34 | −0.34 pts |
| Compliance | 4.69 | 4.54 | −0.15 pts |
| Quality Measures | 3.52 | 3.51 | ≈ 0 |

Staffing drives almost the entire gap. Compliance and health outcome measures (Quality Measures) are nearly identical across ownership types. This means for-profit facilities are not delivering worse *health outcomes* per se — they are delivering fewer staff hours. The mandate (Oct 2023) was designed to close exactly this gap, but the trend line shows the ownership gap has not meaningfully narrowed since.

---

### Finding 3 — For-profit receives the most government funding per facility, yet scores lowest
In 2024, average government funding per facility by ownership type:
- **For-Profit: $7.46M per facility**
- Not-for-Profit: $5.54M per facility
- Government: $2.85M per facility

For-profit facilities receive 2.6× more public funding than government-run facilities per site, yet score 0.57 points lower on quality. The efficiency gap runs in the opposite direction to what the "market competition improves quality" argument would predict.

---

### Finding 4 — Metro areas are the most privatised and the lowest quality
For-profit facilities make up **43.6% of MM1 major city facilities** — the single highest concentration of any remoteness band. This is why the map in Chapter 1 shows patches of low quality in metropolitan areas: it's not that cities have worse aged care infrastructure, it's that they have more for-profit operators.

By contrast, MM6 and MM7 (remote and very remote) have 0% for-profit penetration — which is why remote facilities, counterintuitively, score *higher* on quality than city ones.

---

### Finding 5 — The gap is consistent, not closing
The quarterly trend (May 2023 → Feb 2026) shows the ownership quality gap has remained stable across all 12 snapshots. The October 2023 staffing mandate lifted all ownership types, but did not compress the gap. Government facilities improved; for-profit improved by roughly the same amount. The structural gap between them has not closed.

---

### Framing by audience

| Audience | Message |
|----------|---------|
| **For families choosing a facility** | Who owns the facility is the strongest single quality signal available. Government-run and NFP facilities outperform for-profit across every geography and every year in the dataset. |
| **For families in rural areas** | If your family member is in a regional or remote facility, it is almost certainly run by a charity or government — and it is likely performing better than a city competitor. Distance is not the disadvantage; ownership is. |
| **For policymakers** | The staffing mandate improved staffing scores across all types, but the for-profit compliance gap remains wide (Chapter 4). Funding allocation does not track quality: for-profit receives more per facility, not less. |
| **For investors entering the sector** | Metro markets are 43.6% for-profit and showing lower quality — there is a real opportunity to compete on care standards. The data also shows that higher per-facility funding does not automatically produce better outcomes; operational model matters more. |